# MediPilot — M06 LLM structurer bake-off (Kaggle T4)

**What this decides:** which LLM turns a patient's sentence into structured fields, and whether
an open-weight model that could run inside the hospital is good enough to replace the hosted one.

M06 (`intake/llm_structurer.py`) is extraction *only*. Its output schema has no property for a
diagnosis, an acuity or a band — the model reports what was said, and a fixed table
(`intake/red_flags.py`) decides what it means (§10). This notebook measures both halves: how well
each model extracts, and whether the perception-only guarantee holds under pressure.

---

## Setup

1. **Settings → Accelerator → GPU T4 x2.**
2. **Upload `medipilot-model/` as a Kaggle Dataset** and add it via *+ Add Input*.
3. *(For the hosted comparison)* **Add-ons → Secrets → `GROQ_API_KEY`.**

## How to read the result

Rank on **`red_flags_missed`** first, then `symptom_f1`. A model with the best F1 and one missed
red flag loses to a model with worse F1 and none — under Invariant 1 escalation costs one
assessment and de-escalation can cost a life, so those two errors are never averaged together.

**`schema_failures` and `forbidden_key_hits` are disqualifying, not weighted.** One leaked band
means the perception-only guarantee did not hold for that model.

The `RuleBasedStructurer` row is the **floor**, not a candidate: it is a keyword matcher, and on
the 40-case set it scores roughly `symptom_f1 ≈ 0.51` with `12` missed red flags and `3` spurious.
An LLM that does not clearly beat that — particularly on the `negation` cases, where the baseline
extracts *chest_pain* from *"I have no chest pain"* — is not worth the API call.

## 1 · Environment check

T4s are Turing and do **not** support bfloat16. Everything below uses float16 or 4-bit NF4.
A model card that says `torch_dtype=torch.bfloat16` will crash here.

In [ ]:
import subprocess, torch
print(subprocess.run(['nvidia-smi','--query-gpu=name,memory.total','--format=csv'],
                     capture_output=True, text=True).stdout)
print('torch', torch.__version__, '| cuda', torch.cuda.is_available())
print('bf16 supported:', torch.cuda.is_bf16_supported(), '(False on T4 — use fp16/int4)')

## 2 · Install

`xgrammar` gives real JSON-schema-constrained decoding for the local models. That matters more
than it looks: constrained decoding is what makes the perception-only guarantee **structural**
rather than a matter of the model choosing to obey a prompt. If the schema has no `band`
property, the model cannot emit one — it is not being trusted not to.

If `xgrammar` fails to install, the adapter below falls back to prompt-only JSON and every such
run is labelled `[unconstrained]`. Those rows are still informative, but they are not the same
experiment, and the difference between them is itself a result worth reporting.

In [ ]:
%pip install -q transformers accelerate bitsandbytes groq 2>&1 | tail -2
%pip install -q xgrammar 2>&1 | tail -2
import importlib
HAS_XGRAMMAR = importlib.util.find_spec('xgrammar') is not None
print('constrained decoding available:', HAS_XGRAMMAR)

## 3 · Find the repo and validate the eval set

In [ ]:
import os, sys, shutil

def find_repo():
    # A shallow glob is not enough: Kaggle sometimes mounts a dataset at
    # /kaggle/input/datasets/<owner>/<slug>/... (one level deeper than the
    # classic /kaggle/input/<slug>/...) depending on how it was attached.
    # os.walk with a depth cap is robust to either layout.
    for search_root in ('/kaggle/input', '/kaggle/working', '.', '..'):
        if not os.path.isdir(search_root):
            continue
        for dirpath, dirnames, _ in os.walk(search_root):
            rel = os.path.relpath(dirpath, search_root)
            depth = 0 if rel == '.' else rel.count(os.sep) + 1
            if depth > 6:
                dirnames[:] = []
                continue
            if 'eval' in dirnames and 'intake' in dirnames:
                return os.path.abspath(dirpath)
    raise SystemExit('Upload medipilot-model/ as a Kaggle Dataset (must contain eval/ and intake/).')

REPO = find_repo()
if REPO.startswith('/kaggle/input'):
    dst = '/kaggle/working/medipilot-model'
    if not os.path.exists(dst):
        shutil.copytree(REPO, dst)
    REPO = dst
sys.path.insert(0, REPO)
os.chdir(REPO)
print('repo:', REPO)

The label check below is not a formality. It re-derives every case's expected red-flag rules by
running `intake/red_flags.py` over that case's expected symptoms, and fails if the two disagree.
That guards the §10 split: the eval set must never assert a red-flag expectation the real fixed
table would not produce, or the bake-off would be scoring models against a rule the system does
not have.

In [ ]:
from eval import metrics
from eval.run_structurer_bakeoff import (
    load_cases, check_labels, run_candidate, SUMMARY_COLUMNS, GROQ_CANDIDATES,
)

cases = load_cases()
assert check_labels(cases) == 0, 'eval set labels are inconsistent with the fixed table'

from collections import Counter
print('\ncase families:', dict(Counter(t for c in cases for t in c.get('tags', []))))

## 4 · Baseline — the floor to beat

`RuleBasedStructurer` is a deterministic keyword matcher shipped in `intake/llm_structurer.py`.
It is not a candidate for the job; it is the number that tells you whether an LLM is earning its
place. Watch the `negation` family in particular.

In [ ]:
from intake.llm_structurer import RuleBasedStructurer

summaries = [run_candidate('RuleBasedStructurer (baseline, not an LLM)',
                           RuleBasedStructurer(), cases, verbose=True)]

## 5 · Local open-weight candidates

The adapter below implements the same `LLMStructurer` interface as the production Groq
structurer and reuses `validate_structured_narrative()` verbatim — the same defence-in-depth
check that rejects forbidden clinical-decision keys. Nothing about the scoring is special-cased
for local models.

**Why open-weight models are worth measuring at all:** under DPDP 2023 and the §13 log properties
(*"Raw patient data stays at the institution"*), a structurer that runs inside the hospital is
architecturally preferable to one that sends every patient utterance to a third-party API. The
hosted path is the prototype's choice for demo robustness (§16); this section measures the price
of the alternative, so the V1 conversation with a partner hospital can start from evidence.

In [ ]:
import gc, json, torch
from transformers import AutoModelForCausalLM, AutoTokenizer, BitsAndBytesConfig

from intake.llm_structurer import (
    LLMStructurer, StructurerOutputError, _SCHEMA, _SYSTEM_PROMPT,
    validate_structured_narrative, _empty_narrative,
)


class LocalHFStructurer(LLMStructurer):
    """M06 backed by a local HF causal LM, schema-constrained via xgrammar
    where available. Same interface, same schema, same validator as the
    production Groq structurer."""

    def __init__(self, model_id, load_in_4bit=True, max_new_tokens=512):
        self.model_id = model_id
        self.max_new_tokens = max_new_tokens
        quant = BitsAndBytesConfig(
            load_in_4bit=True, bnb_4bit_quant_type='nf4',
            # float16, never bfloat16 — T4 is Turing.
            bnb_4bit_compute_dtype=torch.float16,
        ) if load_in_4bit else None
        self.tok = AutoTokenizer.from_pretrained(model_id)
        self.model = AutoModelForCausalLM.from_pretrained(
            model_id, quantization_config=quant, torch_dtype=torch.float16,
            device_map='auto',
        )
        self.model.eval()
        self.compiled_grammar = self._compile_grammar()

    def _compile_grammar(self):
        """Compile the M06 JSON schema into a decoding grammar ONCE.

        The LogitsProcessor built from it is stateful -- it tracks position
        in the grammar as tokens are emitted -- so a fresh one is created
        per generation in structure() rather than reused. A processor shared
        across cases would carry the previous case's parse state and
        silently corrupt every result after the first, which is the kind of
        bug that produces a plausible-looking table of wrong numbers.
        """
        if not HAS_XGRAMMAR:
            return None
        try:
            import xgrammar as xgr
            info = xgr.TokenizerInfo.from_huggingface(
                self.tok, vocab_size=self.model.config.vocab_size)
            return xgr.GrammarCompiler(info).compile_json_schema(json.dumps(_SCHEMA))
        except Exception as e:
            print(f'  xgrammar unavailable for {self.model_id}: {e}')
            return None

    @property
    def label(self):
        suffix = '' if self.compiled_grammar is not None else ' [unconstrained]'
        return f'local:{self.model_id}{suffix}'

    def structure(self, transcript, context=None):
        text = (transcript or '').strip()
        if not text:
            return _empty_narrative(transcript or '', 'empty_input')

        messages = [
            {'role': 'system', 'content': _SYSTEM_PROMPT +
             '\n\nRespond with a single JSON object matching this schema and nothing else:\n'
             + json.dumps(_SCHEMA)},
            {'role': 'user', 'content': f'Transcript of one intake turn:\n\n{text}'},
        ]
        prompt = self.tok.apply_chat_template(messages, tokenize=False,
                                              add_generation_prompt=True)
        inputs = self.tok(prompt, return_tensors='pt').to(self.model.device)

        kwargs = dict(max_new_tokens=self.max_new_tokens, do_sample=False,
                      pad_token_id=self.tok.eos_token_id)
        if self.compiled_grammar is not None:
            import xgrammar as xgr
            # Fresh, unused processor for every generation -- see _compile_grammar().
            kwargs['logits_processor'] = [
                xgr.contrib.hf.LogitsProcessor(self.compiled_grammar)]

        with torch.no_grad():
            out = self.model.generate(**inputs, **kwargs)
        completion = self.tok.decode(out[0][inputs['input_ids'].shape[1]:],
                                     skip_special_tokens=True).strip()

        try:
            raw = json.loads(self._extract_json(completion))
        except Exception as exc:
            # Counted as a schema_failure by the runner. Not smoothed over:
            # a model that cannot reliably emit the schema is a model whose
            # extraction silently disappears on some turns.
            raise StructurerOutputError(f'malformed structurer output: {exc}') from exc
        return validate_structured_narrative(raw, text)

    @staticmethod
    def _extract_json(completion):
        """Unconstrained models wrap JSON in prose or fences. Pull out the
        first balanced object rather than failing on the wrapper — the
        experiment is about extraction quality, not markdown discipline."""
        s = completion.strip()
        if s.startswith('```'):
            s = s.split('```')[1]
            s = s[4:] if s.startswith('json') else s
        start = s.find('{')
        if start < 0:
            return s
        depth = 0
        for i, ch in enumerate(s[start:], start):
            depth += (ch == '{') - (ch == '}')
            if depth == 0:
                return s[start:i + 1]
        return s[start:]

### Candidates

**The shortlist is chosen for where these models have to end up, not for where they are being
tested.** Kaggle's T4 is a measuring instrument; the winner has to run on the demo machine — a
Ryzen 5 5600H, 16 GB RAM, **AMD Radeon RX 6500M with 4 GB VRAM and no CUDA**. A 7B model in 4-bit
needs roughly 4.5 GB of weights alone and will not fit that GPU, so it would fall back to CPU,
where it is too slow for a kiosk that has to answer between questions.

So the list is split, and the `fits_locally` column in the results says which is which:

| Tier | Size | Where it can run on the target machine |
| --- | --- | --- |
| **local-deployable** | 1B–4B | Ollama or llama.cpp (Vulkan) with partial GPU offload; Q4 weights ≈ 1–2.5 GB. This is the tier the shipped product draws from. |
| **reference** | 7B–9B | T4 only. Included as a ceiling — the gap between the tiers is what going local actually costs, and that number belongs in the report. |

`bitsandbytes` NF4 here is a stand-in for llama.cpp's Q4_K_M on the target: both are 4-bit, so the
quality ranking transfers, but the *runtime* does not. Confirm the winner with Ollama on the actual
laptop before committing — a model that scores well here and takes nine seconds a turn there has
not solved the problem.

In [ ]:
# (model_id, tier). 'local' = expected to run on 4 GB VRAM / 16 GB RAM via
# llama.cpp or Ollama. 'reference' = T4-class, measured to size the gap.
LOCAL_CANDIDATES = [
    ('Qwen/Qwen2.5-3B-Instruct',         'local'),      # ~2.0 GB at Q4; strong multilingual for its size
    ('Qwen/Qwen2.5-1.5B-Instruct',       'local'),      # ~1.1 GB; the 'does a kiosk-class model work at all' rung
    ('microsoft/Phi-4-mini-instruct',    'local'),      # ~2.5 GB; unusually good at structured output
    ('meta-llama/Llama-3.2-3B-Instruct', 'local'),      # gated: accept the licence on HF first
    ('Qwen/Qwen2.5-7B-Instruct',         'reference'),  # the ceiling
    ('meta-llama/Llama-3.1-8B-Instruct', 'reference'),  # gated
]

for model_id, tier in LOCAL_CANDIDATES:
    print(f'\n{"=" * 70}\n=== {model_id}  [{tier}] ===')
    structurer = None
    try:
        structurer = LocalHFStructurer(model_id)
        s = run_candidate(f'{structurer.label} [{tier}]', structurer, cases, verbose=True)
        s['tier'] = tier
        s['fits_locally'] = (tier == 'local')
        summaries.append(s)
    except Exception as e:
        # A gated repo or an OOM should cost you one model, not the whole run.
        print(f'  SKIPPED — {type(e).__name__}: {e}')
    finally:
        del structurer
        gc.collect(); torch.cuda.empty_cache()

## 6 · Hosted candidates — Groq

These use `response_format={'type':'json_schema', strict:True}`, so the schema constraint is
enforced server-side by the same mechanism production uses. This is the row the demo depends on.

In [ ]:
try:
    from kaggle_secrets import UserSecretsClient
    os.environ['GROQ_API_KEY'] = UserSecretsClient().get_secret('GROQ_API_KEY')
    print('GROQ_API_KEY loaded')
except Exception as e:
    print('No Groq secret — skipping hosted candidates.', e)

if os.environ.get('GROQ_API_KEY'):
    from intake.llm_structurer import GroqLLMStructurer
    for model in GROQ_CANDIDATES:
        print(f'\n=== groq:{model} ===')
        try:
            summaries.append(run_candidate(f'groq:{model}', GroqLLMStructurer(model),
                                           cases, verbose=True))
        except Exception as e:
            print(f'  SKIPPED — {e}')

## 7 · Results

In [ ]:
summaries.sort(key=lambda s: (s['forbidden_key_hits'], s['red_flags_missed'],
                              s['schema_failures'], -(s['symptom_f1'] or 0)))

cols = list(SUMMARY_COLUMNS)
if any('fits_locally' in s for s in summaries):
    # Whether a model can run on the target hardware is a property of
    # the candidate, not a metric — and it is the column that decides
    # what actually ships, so it sits next to the name.
    cols.insert(1, 'fits_locally')
table = metrics.to_markdown_table(summaries, cols)
from IPython.display import Markdown, display
display(Markdown('## M06 structurer bake-off\n\n' + table))
print('\nRanked by forbidden-key hits, then missed red flags, then schema failures, '
      'then symptom F1.')

for s in summaries:
    if s['_leaks']:
        print(f"\nPossible clinical-decision leaks — {s['candidate']}:")
        for leak in s['_leaks']:
            print('  -', leak)

### Where each model actually fails

The per-family breakdown is what goes in front of a judge. A model can carry a respectable
average and still fail every negation case, and that single column is the difference between an
LLM structurer and an expensive keyword matcher.

In [ ]:
import pandas as pd

rows = []
for s in summaries:
    by_tag = {}
    for r in s['_rows']:
        for tag in r['tags']:
            by_tag.setdefault(tag, []).append(r)
    entry = {'candidate': s['candidate']}
    for tag, rs in sorted(by_tag.items()):
        entry[f'{tag}_f1'] = round(sum(r['symptom_f1'] for r in rs) / len(rs), 3)
        entry[f'{tag}_rf_miss'] = sum(len(r['red_flag_missed']) for r in rs)
    rows.append(entry)

df = pd.DataFrame(rows).set_index('candidate')
display(df)

print('\nEvery case any candidate got wrong on red flags:')
for s in summaries:
    bad = [r for r in s['_rows'] if r['red_flag_missed'] or r['red_flag_spurious']]
    if bad:
        print(f"\n  {s['candidate']}")
        for r in bad:
            print(f"    {r['id']:<6} missed={r['red_flag_missed']} "
                  f"spurious={r['red_flag_spurious']}  {r['transcript'][:60]!r}")

## 8 · Save

Download both from the notebook's Output tab. `structurer_bakeoff_table.md` pastes straight into
the report; `structurer_bakeoff_results.json` carries every per-case row behind the averages, which
is what you want when a judge asks *"which case did it get wrong?"*

In [ ]:
import json

with open('/kaggle/working/structurer_bakeoff_results.json', 'w', encoding='utf-8') as fh:
    json.dump(summaries, fh, indent=2, ensure_ascii=False, default=str)

with open('/kaggle/working/structurer_bakeoff_table.md', 'w', encoding='utf-8') as fh:
    fh.write('## M06 structurer bake-off\n\n' + table + '\n\n')
    fh.write(df.to_markdown() + '\n')

print('saved to /kaggle/working/')